# 08 · Expected promotion cost per July customer × action
Target = Step 03 `promo_cost_14d` (discount on redeemed PCT orders; inferred $5.99 waived fee on redeemed FREE_SHIP orders < $75.08).
One `XGBRegressor` per commercial action on **exploration** rows learns `E[promo_cost_14d | X, action]`; NO_OFFER cost = 0.
Reuses the Step 06 configuration after checking it on CP06. Final July cost = max(prediction, 0).

## Load data

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np, pandas as pd, joblib
from src.common import TABLES, MODELS, load_processed
from src.modeling import ACTIONS, COMMERCIAL, feature_lists, make_xgb_pipe, reg_metrics
pd.set_option("display.width", 200)
df = load_processed("historical_feature_outcome")
exp = df[df.assignment_source == "exploration"]
assert exp.promo_cost_14d.notna().all(), "promo_cost_14d has missing targets"
NUM, CAT = feature_lists()
X_COLS = NUM + CAT
CFG = json.loads((MODELS / "tlearner_metadata.json").read_text())["xgb_selected"]
jf = load_processed("july_features")
scores = pd.read_csv(TABLES / "july_action_ncm_scores.csv", parse_dates=["policy_as_of_ts"])
SPLIT = {"train": [f"CP0{i}" for i in range(1, 6)], "valid": ["CP06"], "test": ["CP07"]}
print("config reused from Step 06:", CFG)
exp[exp.offer_type.isin(COMMERCIAL)].groupby("offer_type").promo_cost_14d.agg(["size", "mean", lambda s: (s > 0).mean()]).rename(columns={"<lambda_0>": "share_nonzero"}).round(3)

config reused from Step 06: {'n_estimators': 100, 'max_depth': 1, 'learning_rate': 0.03}


,size,mean,share_nonzero
offer_type,,,
FREE_SHIP,1906,0.101,0.017
PCT_10,1902,4.683,0.121
PCT_20,1893,8.887,0.124


## Validate on CP06 (model vs action-mean baseline), test on CP07, refit CP01–07

In [2]:
part = lambda a, k: exp[(exp.offer_type == a) & exp.campaign_id.isin(sum((SPLIT[x] for x in k), []))]
rows, final = [], {}
for a in COMMERCIAL:
    tr, va, te, trva = part(a, ["train"]), part(a, ["valid"]), part(a, ["test"]), part(a, ["train", "valid"])
    m = make_xgb_pipe(NUM, CAT, CFG).fit(tr[X_COLS], tr.promo_cost_14d)
    rows.append({"action": a, "split": "valid_CP06", **reg_metrics(va.promo_cost_14d, np.maximum(m.predict(va[X_COLS]), 0))})
    rows.append({"action": a, "split": "valid_CP06_mean_baseline", **reg_metrics(va.promo_cost_14d, np.full(len(va), tr.promo_cost_14d.mean()))})
    m = make_xgb_pipe(NUM, CAT, CFG).fit(trva[X_COLS], trva.promo_cost_14d)
    rows.append({"action": a, "split": "test_CP07", **reg_metrics(te.promo_cost_14d, np.maximum(m.predict(te[X_COLS]), 0))})
    final[a] = make_xgb_pipe(NUM, CAT, CFG).fit(part(a, ["train", "valid", "test"])[X_COLS], part(a, ["train", "valid", "test"]).promo_cost_14d)
metrics = pd.DataFrame(rows)
metrics.to_csv(TABLES / "promo_cost_metrics.csv", index=False)
metrics.round(3)

,action,split,n,rmse,mae,r2,obs_mean,pred_mean
0,PCT_10,valid_CP06,285,13.630,6.923,0.037,2.976,5.248
1,PCT_10,valid_CP06_mean_baseline,285,14.033,6.993,-0.021,2.976,5.001
2,PCT_10,test_CP07,294,20.894,8.288,-0.006,4.905,4.594
3,PCT_20,valid_CP06,266,29.439,15.108,0.047,9.734,8.722
4,PCT_20,valid_CP06_mean_baseline,266,30.185,15.503,-0.002,9.734,8.374
5,PCT_20,test_CP07,243,31.283,16.432,0.035,10.879,8.541
6,FREE_SHIP,valid_CP06,283,0.373,0.122,-0.101,0.021,0.101
7,FREE_SHIP,valid_CP06_mean_baseline,283,0.366,0.131,-0.063,0.021,0.110
8,FREE_SHIP,test_CP07,266,0.897,0.229,-0.017,0.135,0.098


## Save cost models and reload check

In [3]:
for a, m in final.items():
    joblib.dump(m, MODELS / f"promo_cost_{a}.joblib")
cost_models = {a: joblib.load(MODELS / f"promo_cost_{a}.joblib") for a in COMMERCIAL}
meta = {"target": "promo_cost_14d (Step 03 definition)", "training_population": "exploration rows of each commercial action, CP01-CP07",
        "xgb_config": CFG, "clip": "estimated_promo_cost = max(prediction, 0)", "NO_OFFER": "deterministic 0",
        "train_rows": {a: int((exp.offer_type == a).sum()) for a in COMMERCIAL}}
(MODELS / "promo_cost_metadata.json").write_text(json.dumps(meta, indent=1))
print("saved + reloaded:", [f"promo_cost_{a}.joblib" for a in COMMERCIAL])

saved + reloaded: ['promo_cost_PCT_10.joblib', 'promo_cost_PCT_20.joblib', 'promo_cost_FREE_SHIP.joblib']


## Build the long July customer × action value table

In [4]:
X_july = scores[["customer_id"]].merge(jf, on="customer_id", how="left", validate="1:1")
cost = {"NO_OFFER": np.zeros(len(scores))}
raw = {a: cost_models[a].predict(X_july[X_COLS]) for a in COMMERCIAL}
cost.update({a: np.maximum(raw[a], 0) for a in COMMERCIAL})
long = pd.concat([scores[["customer_id", "campaign_id", "policy_as_of_ts", "predicted_purchase_probability"]].assign(
    action=a, predicted_ncm=scores[f"predicted_ncm_{a}"].values, estimated_incremental_ncm=scores[f"incremental_ncm_{a}"].values,
    estimated_promo_cost=cost[a]) for a in ACTIONS], ignore_index=True)
long["action"] = pd.Categorical(long.action, ACTIONS)
long = long.sort_values(["customer_id", "action"]).reset_index(drop=True)
long["action"] = long.action.astype(str)
long = long[["customer_id", "campaign_id", "policy_as_of_ts", "action", "predicted_purchase_probability", "predicted_ncm",
             "estimated_incremental_ncm", "estimated_promo_cost"]]
num = long[["predicted_ncm", "estimated_incremental_ncm", "estimated_promo_cost"]].to_numpy(float)
no = long.action == "NO_OFFER"
checks = {"four action rows per customer": long.groupby("customer_id").action.nunique().eq(4).all() and long.groupby("customer_id").size().eq(4).all(),
          "row count = 4 x scoring customers": len(long) == 4 * scores.customer_id.nunique(),
          "valid action names only": set(long.action) == set(ACTIONS),
          "finite NCM / incremental / cost": np.isfinite(num).all(),
          "no negative cost": (long.estimated_promo_cost >= 0).all(),
          "NO_OFFER cost = 0 and incremental = 0": (long.loc[no, ["estimated_promo_cost", "estimated_incremental_ncm"]] == 0).all().all()}
print(pd.Series(checks).to_string()); assert all(checks.values())
print("raw predictions clipped at 0:", {a: int((raw[a] < 0).sum()) for a in COMMERCIAL})
long.to_csv(TABLES / "july_customer_action_values.csv", index=False)
long.groupby("action").agg(mean_cost=("estimated_promo_cost", "mean"), total_cost_if_all=("estimated_promo_cost", "sum"),
                           mean_incremental_ncm=("estimated_incremental_ncm", "mean")).reindex(ACTIONS).round(3)

four action rows per customer            True
row count = 4 x scoring customers        True
valid action names only                  True
finite NCM / incremental / cost          True
no negative cost                         True
NO_OFFER cost = 0 and incremental = 0    True
raw predictions clipped at 0: {'PCT_10': 0, 'PCT_20': 0, 'FREE_SHIP': 0}


,mean_cost,total_cost_if_all,mean_incremental_ncm
action,,,
NO_OFFER,0.000,0.000,0.000
PCT_10,4.279,11877.561,-2.858
PCT_20,8.550,23734.835,-6.449
FREE_SHIP,0.089,245.936,2.184
